In [ ]:
# ==============================================================================
# 1. CONFIGURAÇÃO DE REPRODUTIBILIDADE & BIBLIOTECAS
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)

In [ ]:
# ==============================================================================
# 2. CARREGAMENTO DOS DADOS
# ==============================================================================
url_raw_github = 'https://raw.githubusercontent.com/marquesdanielb/dados_musicais/refs/heads/main/treino_avaliacao.csv'

print('Iniciando a carga dos dados...')
df = pd.read_csv(url_raw_github)
print(f'Sucesso! O dataset possui {df.shape[0]} linhas (músicas) e {df.shape[1]} colunas (atributos).')

In [ ]:
# ==============================================================================
# 3. ANÁLISE EXPLORATÓRIA DE DADOS (EDA)
# ==============================================================================
print('*** Visão Geral do Schema ***')
df.info()

print('\n*** Amostra dos Dados ***')
display(df.head(3))

# Contagem absoluta e percentual de cada gênero
contagem_generos = df['Gênero'].value_counts()
percentual_generos = df['Gênero'].value_counts(normalize=True) * 100

print('\n*** Distribuição Percentual por Gênero ***')
print(percentual_generos)

# Plotando o gráfico de barras
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y='Gênero', order=contagem_generos.index, 
              hue='Gênero', legend=False, palette='viridis')
plt.title('Distribuição de Músicas por Gênero Musical (Target)')
plt.xlabel('Quantidade de Músicas')
plt.ylabel('Gênero')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Resumo Estatístico das Emoções
colunas_emocoes = ['Raiva', 'Medo', 'Alegria', 
                   'Tristeza', 'Surpresa', 'Valor']
print('\n*** Resumo Estatístico das Variáveis Numéricas ***')
display(df[colunas_emocoes].describe())

In [ ]:
# ==============================================================================
# 4. PREPARAÇÃO E DIVISÃO DOS DADOS
# ==============================================================================
colunas_para_remover = ['Unnamed: 0', 'Nome da Música', 'ID do Artista', 
                        'Letra Tratada', 'Valor', 'Gênero']

X = df.drop(columns=colunas_para_remover)
y = df['Gênero']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print('*** Validação da Divisão dos Dados ***')
print(f'Tamanho do Treino (X_train): {X_train.shape[0]} linhas e {X_train.shape[1]} colunas.')
print(f'Tamanho do Teste (X_test): {X_test.shape[0]} linhas e {X_test.shape[1]} colunas.')


print('\n*** Proporção da classe "funk_carioca" (Estratificação) ***' )
print(f'No treino: {(y_train.value_counts(normalize=True)['funk carioca']*100):.4f}%')
print(f'No teste: {(y_test.value_counts(normalize=True)['funk carioca']*100):.4f}%')

In [ ]:
# ==============================================================================
# 5. MODELAGEM: BASELINE INGÊNUO
# ==============================================================================
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print('*** Relatório de Desempenho do baseline ***')
print(classification_report(y_test, y_pred_baseline, zero_division=0))

In [ ]:
# ==============================================================================
# 6. MODELAGEM: COMPARAÇÃO COM VALIDAÇÃO CRUZADA
# ==============================================================================
modelo_lr = LogisticRegression(max_iter=1000, random_state=SEED)
modelo_rf = RandomForestClassifier(random_state=SEED)

print('Iniciando treinamento com validação cruzada...')

scores_lr = cross_val_score(modelo_lr, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
print(f'Regressão Logística - F1-Macro Médio (CV): {scores_lr.mean():.4f}')

scores_rf = cross_val_score(modelo_rf, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
print(f'Random Forest - F1-Macro Médio (CV): {scores_rf.mean():.4f}')

In [ ]:
# ==============================================================================
# 7. OTIMIZAÇÃO DE HIPERPARÂMETROS
# ==============================================================================
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None]
}

print('Iniciando o GridSearchCV...')
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=SEED),
                           param_grid=param_grid, cv=3, scoring='f1_macro', 
                           n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f'\nMelhores Hiperparâmetros: {grid_search.best_params_}')
print(f'Melhor F1-Macro após Otimização: {grid_search.best_score_:.4f}') 

In [ ]:
# ==============================================================================
# 8. AVALIAÇÃO FINAL E MATRIZ DE CONFUSÃO
# ==============================================================================
melhor_rf = grid_search.best_estimator_
y_pred_final = melhor_rf.predict(X_test)

print('**************************************************************')
print('       Relatório de classificação final (TESTE)               ')
print('**************************************************************')
print(classification_report(y_test, y_pred_final, zero_division=0))

plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred_final, labels=melhor_rf.classes_)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=melhor_rf.classes_,
            yticklabels=melhor_rf.classes_)

plt.title('Matriz de confusão Final - Classificador de Gêneros Musicas')
plt.xlabel('Gênero Predito pelo Modelo')
plt.ylabel('Gênero Real da Música')
plt.tight_layout()
plt.show()